In [ ]:
megadescriptor_version = 'T-224'  # 'S-224', 'B-224', 'L-384'
detection = '' # _detected', '_detected_manual'

In [ ]:
import torch

# Load the saved embeddings
embeddings = f'saved_models/{megadescriptor_version}/embeddings/emb{detection}.pt'
labels = f'saved_models/{megadescriptor_version}/labels/labels{detection}.pt'
label_encoder = f'saved_models/{megadescriptor_version}/label_encoders/label_encoder{detection}.pkl'

all_embeddings = torch.load(embeddings)

print(all_embeddings.shape)  # torch.Size([260, 768])


torch.Size([319, 768])


In [ ]:
import torch, joblib
label_ids = torch.load(labels, weights_only=False)
encoder = joblib.load(label_encoder)

# Convert names back later:
names = encoder.inverse_transform(label_ids)


In [4]:
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

# Example setup
X = embeddings.float()
y = torch.from_numpy(label_ids).long()   # shape (260,)

dataset = TensorDataset(X, y)
train_loader = DataLoader(dataset, batch_size=16, shuffle=True)


In [5]:
class Classifier(nn.Module):
    def __init__(self, input_dim=768, num_classes=10, hidden_dim=256, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, x):
        return self.net(x)


In [6]:
from proportional_split_xy import proportional_split_xy

# Triplet Loss


In [7]:
margin = 0.85

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


### Hard split

In [9]:
import torch
import numpy as np

def generate_hard_triplets(embeddings, labels, num_triplets_per_anchor=20, margin=1.0, device=None):
    """
    Hard triplet mining: for each anchor, pick hardest positive and hardest negative.

    Args:
        embeddings (torch.Tensor): shape (N, D)
        labels (torch.Tensor | np.ndarray): shape (N,)
        num_triplets_per_anchor (int): how many copies to make per anchor
        margin (float): triplet loss margin
        device (torch.device): e.g., torch.device('cuda') or ('cpu')
    """
    if device is None:
        device = torch.device("cpu")

    # ensure tensor
    if isinstance(labels, np.ndarray):
        labels = torch.from_numpy(labels)

    embeddings = embeddings.to(device)
    labels = labels.to(device)
    triplets = []

    # normalize embeddings for cosine-style distance
    embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)

    # pairwise Euclidean distances
    with torch.no_grad():
        dists = torch.cdist(embeddings, embeddings, p=2)

    labels_np = labels.cpu().numpy()
    n = len(labels_np)

    for anchor_idx in range(n):
        anchor_label = labels_np[anchor_idx]

        pos_indices = np.where(labels_np == anchor_label)[0]
        neg_indices = np.where(labels_np != anchor_label)[0]

        if len(pos_indices) < 2:
            continue  # skip if only one image of this class

        # hardest positive = most distant same-class example
        pos_dists = dists[anchor_idx, pos_indices]
        hardest_pos_idx = pos_indices[torch.argmax(pos_dists)].item()

        # hardest negative = closest different-class example
        neg_dists = dists[anchor_idx, neg_indices]
        hardest_neg_idx = neg_indices[torch.argmin(neg_dists)].item()

        # replicate triplet if needed
        for _ in range(num_triplets_per_anchor):
            triplets.append((anchor_idx, hardest_pos_idx, hardest_neg_idx))

    print(f"✅ Generated {len(triplets)} hard triplets on device {device}.")
    return triplets


### Semi hard split

In [10]:
def generate_semi_hard_triplets(embeddings, labels, num_triplets_per_anchor=20, margin=1.0, device=None):
    """
    Semi-hard triplet mining:
    For each anchor, choose positive that is closer than negative but still violates the margin slightly:
        d(a, p) < d(a, n) < d(a, p) + margin

    embeddings: torch.Tensor (N, D)
    labels: torch.Tensor (N,)
    """
    if isinstance(labels, np.ndarray):
        labels = torch.from_numpy(labels)
    if device is not None:
        embeddings = embeddings.to(device)
        labels = labels.to(device)

    triplets = []
    num_samples = len(labels)

    # Precompute pairwise distances
    dists = torch.cdist(embeddings, embeddings, p=2).detach().cpu()

    labels_np = labels.cpu().numpy()

    for anchor_idx in range(num_samples):
        anchor_label = labels_np[anchor_idx]

        pos_indices = np.where(labels_np == anchor_label)[0]
        neg_indices = np.where(labels_np != anchor_label)[0]
        if len(pos_indices) < 2 or len(neg_indices) == 0:
            continue

        # Hardest positive (furthest same-class)
        d_ap_all = dists[anchor_idx, pos_indices]
        d_ap_all = d_ap_all[d_ap_all > 0]  # exclude self
        if len(d_ap_all) == 0:
            continue

        for _ in range(num_triplets_per_anchor):
            pos_idx = np.random.choice(pos_indices)
            d_ap = dists[anchor_idx, pos_idx]

            # Find semi-hard negatives
            d_an_candidates = dists[anchor_idx, neg_indices]
            mask = (d_an_candidates > d_ap) & (d_an_candidates < d_ap + margin)
            valid_negatives = neg_indices[mask.numpy()]

            if len(valid_negatives) == 0:
                # fallback to hardest negative
                neg_idx = neg_indices[d_an_candidates.argmin().item()]
            else:
                neg_idx = np.random.choice(valid_negatives)

            triplets.append((anchor_idx, pos_idx, neg_idx))

    print(f"Generated {len(triplets)} semi-hard triplets.")
    return triplets


### Random split

In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import defaultdict

# ------------------------------
# Model: scales each embedding dimension independently
# ------------------------------
class ScaledEmbeddingModel(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(emb_dim))
    def forward(self, x):
        return x * self.scale


# ------------------------------
# Compute accuracy: fraction of triplets where anchor-positive < anchor-negative
# ------------------------------
def compute_triplet_accuracy(model, embeddings, triplets):
    correct = 0
    with torch.no_grad():
        for (a, p, n) in triplets:
            anchor = model(embeddings[a].unsqueeze(0))
            positive = model(embeddings[p].unsqueeze(0))
            negative = model(embeddings[n].unsqueeze(0))
            dist_pos = torch.norm(anchor - positive, p=2)
            dist_neg = torch.norm(anchor - negative, p=2)
            if dist_pos < dist_neg:
                correct += 1
    return correct / len(triplets)


# ------------------------------
# Stratified random split of triplets
# - ensures disjoint train/test triplets
# - ensures each class (anchor label) appears in both sets
# ------------------------------
def stratified_triplet_split(triplets, labels, test_ratio=0.2, seed=42):
    random.seed(seed)
    class_to_triplets = defaultdict(list)

    # group triplets by anchor class
    for t in triplets:
        a, _, _ = t
        class_to_triplets[labels[a].item()].append(t)

    train, test = [], []

    for cls, cls_triplets in class_to_triplets.items():
        random.shuffle(cls_triplets)
        n_test = max(1, int(len(cls_triplets) * test_ratio))
        test.extend(cls_triplets[:n_test])
        train.extend(cls_triplets[n_test:])

    random.shuffle(train)
    random.shuffle(test)
    return train, test




In [12]:
# ------------------------------
# Training loop
# ------------------------------
def train_triplet_loss(embeddings, train_triplets, val_triplets,
                       emb_dim=128, lr=1e-3, margin=1.0, epochs=50):
    model = ScaledEmbeddingModel(emb_dim)
    criterion = nn.TripletMarginLoss(margin=margin)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        if epoch != 0:
            model.train()
        total_loss = 0.0

        for (a, p, n) in train_triplets:
            anchor = model(embeddings[a].unsqueeze(0))
            positive = model(embeddings[p].unsqueeze(0))
            negative = model(embeddings[n].unsqueeze(0))

            loss = criterion(anchor, positive, negative)
            if epoch != 0:                
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item()

        model.eval()
        train_acc = compute_triplet_accuracy(model, embeddings, train_triplets)
        val_acc = compute_triplet_accuracy(model, embeddings, val_triplets)

        print(f"Epoch {epoch:02d} | loss: {total_loss/len(train_triplets):.6f} | "
              f"train_acc: {train_acc:.3f} | val_acc: {val_acc:.3f}")

    print("\n✅ Final Validation Accuracy:", 
          round(compute_triplet_accuracy(model, embeddings, val_triplets), 4))
    return model

### Main

In [ ]:
if __name__ == "__main__":
    import random
    import numpy as np

    torch.manual_seed(0)
    random.seed(0)

    # --- Load your embeddings and labels ---
    
    labels = torch.load("saved_models\\T_224\\labels\\labels_povodne.pt", weights_only=False)               # shape (N,)
    N, D = embeddings.shape

    # --- Split into train/val indices ---
    indices = np.arange(N)
    np.random.shuffle(indices)
    split = int(0.8 * N)
    train_idx = indices[:split]
    val_idx = indices[split:]

    def generate_triplets(indices, labels, num_triplets_per_anchor=20):
        triplets = []
        # Handle both torch.Tensor and np.ndarray
        labels_np = labels.cpu().numpy() if isinstance(labels, torch.Tensor) else np.array(labels)

        for anchor_idx in indices:
            anchor_label = labels_np[anchor_idx]

            # Positive samples (same class, excluding anchor)
            pos_indices = [i for i in indices if labels_np[i] == anchor_label and i != anchor_idx]
            # Negative samples (different class)
            neg_indices = [i for i in indices if labels_np[i] != anchor_label]

            if len(pos_indices) == 0 or len(neg_indices) == 0:
                continue

            for _ in range(num_triplets_per_anchor):
                p = random.choice(pos_indices)
                n = random.choice(neg_indices)
                triplets.append((anchor_idx, p, n))

        return triplets


    # --- Generate triplets for train and val ---
    train_triplets = generate_triplets(train_idx, labels, num_triplets_per_anchor=50)
    val_triplets = generate_triplets(val_idx, labels, num_triplets_per_anchor=50)

    # train_triplets = generate_hard_triplets(embeddings[train_idx], labels[train_idx], num_triplets_per_anchor=150, margin=1.1, device=device)
    # val_triplets   = generate_hard_triplets(embeddings[val_idx], labels[val_idx], num_triplets_per_anchor=50, margin=1.0, device=device)


    print(f"Generated {len(train_triplets)} training triplets and {len(val_triplets)} validation triplets.")

    # --- Train model ---
    model = train_triplet_loss(
        embeddings, train_triplets, val_triplets,
        emb_dim=D, lr=1e-3, epochs=20
    )

    # --- Evaluate final accuracy ---
    final_val_acc = compute_triplet_accuracy(model, embeddings, val_triplets)
    print(f"\n✅ Final Validation Accuracy: {final_val_acc:.3f}")

    # --- Optionally save calibrated embeddings ---
    calibrated_embeddings = model(embeddings)
    torch.save(calibrated_embeddings, "emb_calibrated.pt")


Generated 12750 training triplets and 3100 validation triplets.
Epoch 00 | loss: 2.313353 | train_acc: 0.523 | val_acc: 0.506
Epoch 01 | loss: 1.512177 | train_acc: 0.562 | val_acc: 0.538
Epoch 02 | loss: 0.938223 | train_acc: 0.577 | val_acc: 0.542
Epoch 03 | loss: 0.904706 | train_acc: 0.584 | val_acc: 0.534
Epoch 04 | loss: 0.901293 | train_acc: 0.583 | val_acc: 0.535
Epoch 05 | loss: 0.899409 | train_acc: 0.582 | val_acc: 0.537
Epoch 06 | loss: 0.899257 | train_acc: 0.582 | val_acc: 0.532
Epoch 07 | loss: 0.899759 | train_acc: 0.583 | val_acc: 0.533
Epoch 08 | loss: 0.899186 | train_acc: 0.583 | val_acc: 0.531
Epoch 09 | loss: 0.898759 | train_acc: 0.584 | val_acc: 0.535
Epoch 10 | loss: 0.899310 | train_acc: 0.582 | val_acc: 0.536


KeyboardInterrupt: 